In [ ]:
from typing import Callable
from typing import Iterator
from typing import List
from typing import Tuple
from typing import Type
from typing import Any
from typing import Optional
from typing import Type
import os

import numpy as np
import torch
import torch
import torch.nn as nn


In [7]:
"""Tools for tomographic simulations."""



"""Differentiable transformations.

See the bmad-x repository for accelerator modeling capabilities (https://github.com/bmad-sim/Bmad-X).
"""
import math
import numpy as np
import torch
import torch.nn as nn
import scipy.special


def rotation_matrix(angle: float) -> torch.Tensor:
    _cos = np.cos(angle)
    _sin = np.sin(angle)
    return torch.tensor([[_cos, _sin], [-_sin, _cos]])


def reverse_momentum(x):
    for i in range(0, x.shape[1], 2):
        x[:, i + 1] *= -1.0
    return x


class Transform(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError

    def inverse(self, u: torch.Tensor) -> torch.Tensor:
        raise NotImplementedError


class CompositeTransform(Transform):
    def __init__(self, *transforms) -> None:
        super().__init__()
        self.transforms = nn.Sequential(*transforms)
            
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        u = x
        for transform in self.transforms:
            u = transform(u)
        return u

    def inverse(self, u: torch.Tensor) -> torch.Tensor:
        x = u
        for transform in self.transforms[::-1]:
            x = transform.inverse(x)
        return x
    
    def to(self, device):
        for transform in self.transforms:
            transform.to(device)
        return self
        

class LinearTransform(Transform):
    def __init__(self, matrix: torch.Tensor) -> None:
        super().__init__()
        self.set_matrix(matrix)

    def set_matrix(self, matrix: torch.Tensor) -> None:
        self.matrix = matrix
        self.matrix_inv = torch.linalg.inv(matrix)
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.matmul(x, self.matrix.T)

    def inverse(self, u: torch.Tensor) -> torch.Tensor:
        return torch.matmul(u, self.matrix_inv.T)
    
    def to(self, device):
        self.matrix = self.matrix.to(device)
        return self

def unravel(iterable):
    return itertools.chain.from_iterable(iterable)
    
class MultipoleTransform(Transform):
    """Applies multipole kick.
    
    https://github.com/PyORBIT-Collaboration/PyORBIT3/blob/main/src/teapot/teapotbase.cc    
    
    Parameters
    ----------
    order: int
        The multipole number (0 for dipole, 1 for quad,, 2 for sextupole, etc.).
    strength : float
        Integrated kick strength [m^(-pole)].
    skew : bool
        If True, rotate the magnet 45 degrees.
    """
    def __init__(self, order: int, strength: float, skew: bool = False) -> None:
        super().__init__()
        self.order = order
        self.strength = strength
        self.skew = skew
        
    def forward(self, X: torch.Tensor) -> torch.Tensor:

        U = X.clone()
        
        x = X[:, 0]        
        if X.shape[1] > 2:
            y = X[:, 2]
        else:
            y = 0.0 * X[:, 0]

        ## MPS does not support torch.complex.        
        # z = torch.complex(x, y)
        # zn = torch.pow(z, float(self.order - 1))
        # zn = z ** (self.order - 1)
        # zn_imag = zn.imag
        # zn_real = zn.real

        ## Hard-code a few values of n.
        if self.order == 1:
            zn_real = 1.0
            zn_imag = 1.0
        if self.order == 2:
            zn_real = x
            zn_imag = y
        if self.order == 3:
            zn_real = (x**2) - (y**2)
            zn_imag = (2.0 * x * y)
        elif self.order == 4:
            zn_real = (+x**3) - (3.0 * y**2 * x)
            zn_imag = (-y**3) + (3.0 * x**2 * y)
        elif self.order == 5:
            zn_real = (x**4) - (6.0 * x**2 * y**2) + (y**4)
            zn_imag = (4.0 * x**3 * y) - (4.0 * x * y**3)
        else:
            raise ValueError("MPS-compatible MultipoleTransform requires order <= 5.")

        k = self.strength / scipy.special.factorial(self.order - 1)
        if self.skew:
            U[:, 1] = X[:, 1] + k * zn_imag
            if X.shape[1] > 2:
                U[:, 3] = X[:, 3] + k * zn_real
        else:
            U[:, 1] = X[:, 1] - k * zn_real
            if X.shape[1] > 2:
                U[:, 3] = X[:, 1] + k * zn_imag
        return U

    def inverse(self, u: torch.Tensor) -> torch.Tensor:
        return reverse_momentum(self.forward(reverse_momentum(u)))


class ProjectionTransform(Transform):
    """Computes 1D projection along specified direction."""
    def __init__(self, direction: torch.Tensor) -> None:
        super().__init__()
        self.direction = direction / torch.norm(direction)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sum(x * self.direction, dim=1)[:, None]
        


def forward(
    x: torch.Tensor, 
    transforms: List[Type[nn.Module]],
    diagnostics: List[List[Type[nn.Module]]]
) -> List[List[torch.Tensor]]:
    """Compute projections from initial coordinates and transforms/diagnostic set. 

    Parameters
    ----------
    x : tensor, shape (n, d)
        Input particle coordinates.
    transforms : list[Transform],
        A list of transforms.
    diagnostics : list[list[Diagnostic]],
        diagnostics[i] is a list of diagnostics to apply after transform i.

    Returns
    -------
    list[list[tensor]]
        Predictions for each transform and diagnostic. 
    """
    predictions = []
    for index, transform in enumerate(transforms):
        u = transform(x.clone())
        predictions.append([diagnostic(u) for diagnostic in diagnostics[index]])
    return predictions


class Simulator:
    """Manages forward process simulations."""
    def __init__(
        self,         
        transforms: List[Type[nn.Module]],
        diagnostics: List[List[Type[nn.Module]]],
    ) -> None:
        self.transforms = transforms
        self.diagnostics = diagnostics
            
    def forward(self, x: torch.Tensor) -> List[List[torch.Tensor]]:
        return forward(x, self.transforms, self.diagnostics)



In [8]:

class EntropyEstimator(torch.nn.Module):
    """Estimates negative entropy from samples and/or log probability."""
    def __init__(self, prior: Any = None) -> None:
        super().__init__()
        self.prior = prior

    def forward(self, x: torch.Tensor, log_prob: torch.Tensor = None) -> torch.Tensor:
        raise NotImplementedError


class EmptyEntropyEstimator(EntropyEstimator):
    """Returns zero."""
    def __init__(self, prior: Any = None) -> None:
        super().__init__(prior=prior)

    def forward(self, x: torch.Tensor, log_prob: torch.Tensor = None) -> torch.Tensor:
        return 0.0


class CovarianceEntropyEstimator(EntropyEstimator):
    """Estimates negative entropy from covariance matrix."""
    def __init__(self, prior: Any = None, pad: float = 1.00e-12) -> None:
        if prior is not None:
            raise ValueError("This class cannot estimate relative entropy (prior != None).")
        super().__init__(prior=prior)
        self.pad = pad

    def forward(self, x: torch.Tensor, log_prob: torch.Tensor = None) -> torch.Tensor:
        eps = torch.sqrt(torch.det(torch.cov(x.T)))
        H = -3.0 * np.log(2.0 * np.pi * np.e) - torch.log(eps + self.pad)
        return H


class KNNEntropyEstimator(EntropyEstimator):
    """Estimates negative entropy from k nearest neighbors."""
    def __init__(self, prior: Any = None, k: int = 5) -> None:
        if prior is not None:
            raise ValueError("This class cannot estimate relative entropy (prior != None).")
        super().__init__(prior=prior)
        self.k = k

    def forward(self, x: torch.Tensor, log_prob: torch.Tensor = None) -> torch.Tensor:
        raise NotImplementedError


class MonteCarloEntropyEstimator(EntropyEstimator):
    """Estimates negative entropy from Monte Carlo."""
    def __init__(self, prior: Any = None) -> None:
        super().__init__(prior=prior)

    def forward(self, x: torch.Tensor, log_prob: torch.tensor) -> torch.Tensor:
        H = torch.mean(log_prob)
        if self.prior is not None:
            H = H - torch.mean(self.prior.log_prob(x))
        return H
        


In [9]:

def corrupt(x: np.ndarray, scale: float, rng=None) -> np.ndarray:
    return x + rng.normal(scale=scale, size=x.shape)


def decorrelate(x: np.ndarray, rng=None) -> np.ndarray:
    if x.shape[1] % 2 == 0:
        for i in range(0, d, 2):
            j = 2 * i
            idx = rng.permutation(np.arange(n))
            x[:, j : j + 1] = x[idx, j : j + 1]
    else:
        for i in range(0, d, 1):
            idx = rng.permutation(np.arange(n))
            x[:, j] = x[idx, j]
    return x


def normalize(x: np.ndarray) -> np.ndarray:
    x = x - np.mean(x, axis=0)
    x = x / np.std(x, axis=0)
    return x


def shuffle(x: np.ndarray, rng=None) -> np.ndarray:
    return rng.permutation(x)


class Distribution:
    def __init__(
        self, ndim: int = 2,
        seed: int = None, 
        normalize: bool = False, 
        shuffle: bool = True, 
        decorr: bool = False, 
        noise: float = None,
        shear: Callable = None,
        device: torch.device = None,
    ) -> None:
        self.ndim = ndim
        self.seed = seed
        self.rng = np.random.default_rng(seed)
        self.normalize = normalize
        self.shuffle = shuffle
        self.decorr = decorr
        self.noise = noise
        self.shear = shear
        self.device = device

    def _sample(self, size: int) -> np.ndarray:
        raise NotImplementedError

    def _log_prob(self, x: np.ndarray) -> np.ndarray:
        raise NotImplementedError
        
    def sample_np(self, size: int) -> np.ndarray:
        x = self._sample(int(size))
        if self.shuffle:
            x = shuffle(x, rng=self.rng)
        if self.normalize:
            x = normalize(x)
        if self.noise:
            x = corrupt(x, self.noise, rng=self.rng)
        if self.decorr:
            x = decorrelate(x, rng=self.rng)
        if self.shear:
            sigma_old = np.std(x[:, 0])
            x[:, 0] += self.shear * x[:, 1]
            sigma_new = np.std(x[:, 0])
            x[:, 0] *= (sigma_old / sigma_new)
        return x

    def sample(self, size: int) -> torch.Tensor:
        x = self.sample_np(size)
        x = torch.from_numpy(x)
        x = x.type(torch.float32)
        x = x.to(self.device)
        return x
        
    def log_prob(self, x: torch.Tensor) -> torch.Tensor:
        log_prob = self._log_prob(x.detach().cpu().numpy())
        log_prob = torch.from_numpy(log_prob)
        log_prob = log_prob.type(torch.float32)
        log_prob = log_prob.to(device)
        return log_prob


class EightGaussians(Distribution):
    def __init__(self, **kws) -> None:
        kws["ndim"] = 2
        super().__init__(**kws)
        if self.noise is None:
            self.noise = 0.20

    def _sample(self, size: int) -> np.ndarray:
        theta = 2.0 * np.pi * self.rng.integers(0, 8, size) / 8.0
        x = np.stack([np.cos(theta), np.sin(theta)], axis=-1)
        x *= 1.5
        return x


class Galaxy(Distribution):
    def __init__(self, turns: int = 5, truncate: float = 3.0, **kws) -> None:
        kws["ndim"] = 2
        super().__init__(**kws)
        self.turns = turns
        self.truncate = truncate
        if self.noise is None:
            self.noise = 0.0

    def _sample(self, size: int) -> np.ndarray:
        
        def _rotate(x, theta):
            _x = x[:, 0].copy()
            _y = x[:, 1].copy()
            x[:, 0] = _x * np.cos(theta) + _y * np.sin(theta)
            x[:, 1] = _y * np.cos(theta) - _x * np.sin(theta)
            return x

        # Start with flattened Gaussian distribution.
        x = np.zeros((size, 2))
        x[:, 0] = 1.0 * scipy.stats.truncnorm.rvs(-self.truncate, self.truncate, size=size)
        x[:, 1] = 0.5 * scipy.stats.truncnorm.rvs(-self.truncate, self.truncate, size=size)

        # Apply amplitude-dependent phase advance.
        r = np.linalg.norm(x, axis=1)
        r = r / np.max(r)        
        theta = 2.0 * np.pi * (1.0 + 0.5 * (r **0.25))
        for _ in range(self.turns):
            x = _rotate(x, theta)     

        # Normalize
        x /= np.std(x, axis=0)
        x *= 0.85
        return x


class Gaussian(Distribution):
    def __init__(self, **kws) -> None:
        super().__init__(**kws)

    def _sample(self, size: int) -> np.ndarray:
        return self.rng.normal(size=(size, self.ndim))


class GaussianMixture(Distribution):
    def __init__(
        self, 
        modes: int = 7, 
        xmax: float = 3.0, 
        scale: float = 0.75, 
        shiftscale=True, 
        **kws
    ) -> None:
        super().__init__(**kws)
        self.modes = modes
        self.locs = self.rng.uniform(-xmax, xmax, size=(self.modes, self.ndim))
        self.scales = scale * np.ones(self.modes)
        self.shiftscale = shiftscale
        
    def _sample(self, size: int) -> np.ndarray:
        x = [
            self.rng.normal(loc=loc, scale=scale, size=(size // self.modes, self.ndim))
            for scale, loc in zip(self.scales, self.locs)
        ]
        x = np.vstack(x)
        if self.shiftscale:
            x = x - np.mean(x, axis=0)
            x = x / np.std(x, axis=0)
        return x


class Hollow(Distribution):
    def __init__(self, exp: float = 1.66, **kws) -> None:
        super().__init__(**kws)
        self.exp = exp
        if self.noise is None:
            self.noise = 0.05

    def _sample(self, size: int) -> np.ndarray:
        x = KV(ndim=self.ndim, seed=self.seed).sample_np(size)
        r = self.rng.uniform(0.0, 1.0, size=x.shape[0]) ** (1.0 / (self.exp * self.ndim))
        x *= r[:, None]
        x /= np.std(x, axis=0)
        return x


class KV(Distribution):
    def __init__(self, **kws) -> None:
        super().__init__(**kws)
        if self.noise is None:
            self.noise = 0.05

    def _sample(self, size: int) -> np.ndarray:
        x = self.rng.normal(size=(size, self.ndim))
        x /= np.linalg.norm(x, axis=1)[:, None]
        x /= np.std(x, axis=0)
        return x


class Leaf(Distribution):
    def __init__(self, xmax: float = 2.5, **kws) -> None:
        kws["ndim"] = 2
        super().__init__(**kws)
        if self.noise is None:
            self.noise = 0.010
        self.xmax = xmax

        path = os.path.dirname(os.path.abspath(__file__))
        path = os.path.join(path, "./files/leaf.png")
        self.hist = ski.io.imread(path, as_gray=True)
        self.hist = 1.0 - self.hist
        self.hist = self.hist[::-1, :].T
        self.edges = [np.linspace(-self.xmax, +self.xmax, s + 1) for s in self.hist.shape]

    def _sample(self, size: int) -> np.ndarray:
        hist = self.hist
        edges = self.edges
        
        pdf = hist.ravel()
        idx = np.flatnonzero(pdf)
        pdf = pdf[idx]
        pdf = pdf / np.sum(pdf)
        idx = np.random.choice(idx, size, replace=True, p=pdf)
        idx = np.unravel_index(idx, shape=hist.shape)
        lb = [edges[axis][idx[axis]] for axis in range(hist.ndim)]
        ub = [edges[axis][idx[axis] + 1] for axis in range(hist.ndim)]
        x = np.random.uniform(lb, ub).T
        return x


class Pinwheel(Distribution):
    def __init__(self, **kws) -> None:
        super().__init__(**kws)
        if self.noise is None:
            self.noise = 0.10

    def _sample(self, size: int) -> np.ndarray:
        a = self.rng.normal(loc=1.0, scale=0.25, size=size)
        b = self.rng.normal(scale=0.1, size=size)
        theta = 2.0 * np.pi * self.rng.integers(0, 5, size) / 5.0
        theta = theta + np.exp(a - 1.0)
        x = np.stack(
            [
                a * np.cos(theta) - b * np.sin(theta),
                a * np.sin(theta) + b * np.cos(theta),
            ],
            axis=-1
        )
        x /= np.std(x, axis=0)
        return x


class Rings(Distribution):
    def __init__(self, n_rings: int = 2, decay: float = 0.5, **kws) -> None:
        super().__init__(**kws)
        self.n_rings = n_rings
        self.decay = decay
        if self.noise is None:
            self.noise = 0.15

    def _sample(self, size: int) -> np.ndarray:        
        # Set sphere radii.
        radii = np.linspace(1.0, 0.0, self.n_rings, endpoint=False)[::-1]
                
        # Set equal particle density on each sphere.
        sizes = np.array([sphere_surface_area(d=self.ndim, r=r) for r in radii])
        
        # Make density vary linearly with the radius.
        sizes = sizes * np.linspace(1.0, self.decay, self.n_rings)

        # Scale to correct total particle number.
        sizes = sizes * (size / np.sum(sizes))
        sizes = sizes.astype(int)
        
        # Generate particles on each sphere.
        x = []
        dist = KV(ndim=self.ndim, seed=self.seed)
        for size, radius in zip(sizes, radii):
            x.append(radius * dist.sample(size))
        x = np.vstack(x)
        x /= np.std(x, axis=0)
        return x


class SwissRoll(Distribution):
    def __init__(self, **kws) -> None:
        super().__init__(**kws)
        if self.noise is None:
            self.noise = 0.15

    def _sample(self, size: int) -> np.ndarray:
        t = 1.5 * np.pi * (1.0 + 2.0 * self.rng.uniform(0.0, 1.0, size=size))
        x = np.stack([t * np.cos(t), t * np.sin(t)], axis=-1)
        x /= np.std(x, axis=0)
        return x


class TwoSpirals(Distribution):
    def __init__(self, exp=0.65, **kws) -> None:
        super().__init__(**kws)
        self.exp = exp
        if self.noise is None:
            self.noise = 0.070

    def _sample(self, size: int) -> np.ndarray:
        self.exp = 0.75
        t = 3.0 * np.pi * np.random.uniform(0.0, 1.0, size=size) ** self.exp    
        r = t / 2.0 / np.pi * np.sign(self.rng.normal(size=size))
        t = t + self.rng.normal(size=size, scale=np.linspace(0.0, 1.0, size))
        x = np.stack([-r * np.cos(t), r * np.sin(t)], axis=-1)
        x = x / np.std(x, axis=0)
        return x


class WaterBag(Distribution):
    def __init__(self, **kws) -> None:
        super().__init__(**kws)
        if self.noise is None:
            self.noise = 0.05

    def _sample(self, size: int) -> np.ndarray:
        x = KV(ndim=self.ndim, seed=self.seed).sample_np(size)
        r = self.rng.uniform(0.0, 1.0, size=x.shape[0]) ** (1.0 / self.ndim)
        x *= r[:, None]
        x /= np.std(x, axis=0)
        return x


DISTRIBUTIONS = {
    "eight-gaussians": EightGaussians,
    "galaxy": Galaxy,
    "gaussian": Gaussian,
    "gaussian_mixture": GaussianMixture,
    "hollow": Hollow,
    "kv": KV,
    "leaf": Leaf,
    "pinwheel": Pinwheel,
    "rings": Rings,
    "swissroll": SwissRoll,
    "two-spirals": TwoSpirals,
    "waterbag": WaterBag,
}


def get_distribution(name: str, **kws) -> Distribution:
    return DISTRIBUTIONS[name](**kws)



In [10]:

class GenerativeModel(Distribution, torch.nn.Module):
    """Base class for generative models."""
    def forward(self, z: torch.Tensor, **kws) -> torch.Tensor:
        raise NotImplementedError

    def inverse(self, x: torch.Tensor, **kws) -> torch.Tensor:
        raise NotImplementedError

    def forward_steps(self, z: torch.Tensor) -> List[torch.Tensor]:
        raise NotImplementedError
    
    def inverse_steps(self, x: torch.Tensor) -> List[torch.Tensor]:
        raise NotImplementedError

    def sample_base(self, size: int) -> torch.Tensor:
        raise NotImplementedError

    def dim(self) -> int:
        raise NotImplementedError